In [1]:
import os
import regex as re
from typing import BinaryIO
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from utils import read_range_thread, read_range_process, find_chunk_boundaries, PAT

# %load_ext memory_profiler # if needed to measure mem consumption (%%memit)

# If you want to check out the benefits of Thread and Process Executors, you need to run this notebook using bigger file
# You may use bash command in director ../tests/fixtures: for i in {1..400}; do cat tinystories_sample_5M.txt; done > tinystories_sample_2000M.txt

In [2]:
%%timeit
resulst_sync = []
with open("../tests/fixtures/tinystories_sample_20M.txt", "rb") as f:
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        resulst_sync.append(re.finditer(PAT, chunk))

    for part in resulst_sync:
        for obj in part:
            pass

924 ms ± 8.69 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [3]:
%%timeit
resulst_thread = []
with open("../tests/fixtures/tinystories_sample_20M.txt", "rb") as f:
    num_workers = 12
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    fd = f.fileno()

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        resulst_thread = list(
            executor.map(
                read_range_thread,
                [fd] * (len(boundaries) - 1),
                boundaries[:-1],
                boundaries[1:],
            )
        )
    
    for part in resulst_thread:
        for obj in part:
            pass

925 ms ± 7.81 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit
results_process = []
with open("../tests/fixtures/tinystories_sample_20M.txt", "rb") as f:
    num_workers = 12
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        results_process = list(
            executor.map(
                read_range_process,
                ["../tests/fixtures/tinystories_sample_20M.txt"] * (len(boundaries) - 1),
                boundaries[:-1],
                boundaries[1:],
            )
        )
    for part in results_process:
        for obj in part:
            pass

349 ms ± 12.3 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
